# 01 — Exploratory Data Analysis (Task 1: Sentiment + Spam)

**Goal.** Understand the data before any modelling. Specifically:

1. Confirm shapes, columns, label encoding, missingness.
2. Check label balance in train / val.
3. Read 10 random positives and 10 random negatives — *by eye*, look for spam mixed into both classes (the brief tells us it is there evenly).
4. Look at document-length distribution.
5. Top-50 most-frequent words per declared class, vocabulary size.
6. Form an early hypothesis about what spam *looks like* (URLs? currency symbols? `Subject:`? "click here"?). This informs design choices in `02_baseline_wordlist`, `03_spam_handling`, and `04_models`.

Nothing is saved to disk from this notebook. Output is **understanding**, captured as notes-to-self at the bottom.

Lecture references touched here: W02_L03 (tokenisation, normalisation), W02_L04 (bag-of-words, word-list classifiers, evaluation framing).

In [ ]:
import re
import random
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

DATA_DIR = Path('../data')
TRAIN_CSV = DATA_DIR / 'sentiment_analysis_training_data.csv'
VAL_CSV   = DATA_DIR / 'sentiment_analysis_validation_data.csv'
TEST_CSV  = DATA_DIR / 'sentiment_analysis_test_data.csv'

assert TRAIN_CSV.exists(), TRAIN_CSV
assert VAL_CSV.exists(), VAL_CSV
assert TEST_CSV.exists(), TEST_CSV

## 1. Load and check shapes / columns / dtypes

In [ ]:
train = pd.read_csv(TRAIN_CSV)
val   = pd.read_csv(VAL_CSV)
test  = pd.read_csv(TEST_CSV)

print('train:', train.shape, '| columns:', list(train.columns))
print('val:  ', val.shape,   '| columns:', list(val.columns))
print('test: ', test.shape,  '| columns:', list(test.columns))
print()
print('train dtypes:')
print(train.dtypes)
print()
print('train missing per column:')
print(train.isna().sum())

In [ ]:
train.head(3)

## 2. Label balance

Reminder from the brief: training labels are noisy because spam is mixed evenly into both classes — a `label==1` document is *either* a positive review *or* a spam document mislabelled positive (similarly for `0`). Validation labels we'll trust as a teaching signal but should still check.

In [ ]:
print('train label counts:')
print(train['label'].value_counts(dropna=False))
print('train label proportions:')
print(train['label'].value_counts(normalize=True))
print()
print('val label counts:')
print(val['label'].value_counts(dropna=False))
print('val label proportions:')
print(val['label'].value_counts(normalize=True))

## 3. Read 10 random positives and 10 random negatives

This is the most important cell in the notebook. **Read them.** As you read, ask:

- Is this a movie review or is it spam?
- If spam: what gives it away? `Subject:` header? URLs? Money / currency? Generic marketing language?
- Are spam emails differently structured in the two classes, or do they look the same regardless of label?

In [ ]:
def show_samples(df, label, n=10, max_chars=600, seed=SEED):
    pool = df[df['label'] == label].sample(n=n, random_state=seed)
    for i, (_, row) in enumerate(pool.iterrows(), 1):
        text = str(row['text'])
        snippet = text if len(text) <= max_chars else text[:max_chars] + ' ... [truncated]'
        print(f'--- label={label} | sample {i}/{n} | len={len(text)} chars ---')
        print(snippet)
        print()

In [ ]:
show_samples(train, label=1, n=10)

In [ ]:
show_samples(train, label=0, n=10)

## 4. Document length distributions

Spam emails (especially Enron-style) are often noticeably longer or shorter than a typical movie review, so a length histogram is the cheapest first signal that two distributions are mixed inside each class.

In [ ]:
TOKEN_RE = re.compile(r"[a-zA-Z']+")

def tokenize(text):
    return TOKEN_RE.findall(str(text).lower())

train_tokens = train['text'].map(tokenize)
train_lens = train_tokens.map(len)
train['n_tokens'] = train_lens

print(train.groupby('label')['n_tokens'].describe())

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
bins = np.linspace(0, np.percentile(train_lens, 99), 60)
ax[0].hist(train.loc[train.label == 0, 'n_tokens'], bins=bins, alpha=0.8)
ax[0].set_title('label = 0 (declared negative)')
ax[0].set_xlabel('document length (tokens)')
ax[0].set_ylabel('count')
ax[1].hist(train.loc[train.label == 1, 'n_tokens'], bins=bins, alpha=0.8, color='C1')
ax[1].set_title('label = 1 (declared positive)')
ax[1].set_xlabel('document length (tokens)')
fig.suptitle('Token-length histogram by declared label (clipped at 99th percentile)')
plt.tight_layout()
plt.show()

If the histograms look bimodal — a sharp short-document hump and a broader review hump (or vice-versa) — that's the spam population separating from the review population purely by length.

## 5. Top-50 most-frequent words per declared class

Quick BoW frequency, with NLTK English stopwords removed so the signal isn't drowned by `the / and / of`. We *don't* commit to this preprocessing for modelling — that's `02_*`'s job — we just want a peek.

In [ ]:
try:
    from nltk.corpus import stopwords
    STOP = set(stopwords.words('english'))
except LookupError:
    import nltk
    nltk.download('stopwords')
    from nltk.corpus import stopwords
    STOP = set(stopwords.words('english'))

def top_k_words(token_lists, k=50, drop_stop=True, min_len=2):
    c = Counter()
    for toks in token_lists:
        for t in toks:
            if drop_stop and t in STOP:
                continue
            if len(t) < min_len:
                continue
            c[t] += 1
    return c.most_common(k)

top_pos = top_k_words(train_tokens[train.label == 1], k=50)
top_neg = top_k_words(train_tokens[train.label == 0], k=50)

comparison = pd.DataFrame({
    'pos_word': [w for w, _ in top_pos],
    'pos_count': [c for _, c in top_pos],
    'neg_word': [w for w, _ in top_neg],
    'neg_count': [c for _, c in top_neg],
})
comparison

Words that look out-of-place for movie reviews (`subject`, `enron`, `http`, `email`, `company`, `money`, currency tokens, dates) are spam tells. They should appear in *both* class lists if spam is evenly distributed.

### Words with the biggest frequency *difference* between classes (W02_L04)

The lecture (W02_L04) names this as one way to build word-list classifiers. Here it's a sanity check: if the corpus were clean reviews, this list would be sentiment-loaded (`great / boring / wonderful / waste`). If it's polluted with spam, we may also see spam-vocabulary leaking in.

In [ ]:
from collections import Counter

def class_freq(token_lists):
    c = Counter()
    for toks in token_lists:
        c.update(toks)
    return c

freq_pos = class_freq(train_tokens[train.label == 1])
freq_neg = class_freq(train_tokens[train.label == 0])
n_pos = sum(freq_pos.values())
n_neg = sum(freq_neg.values())

vocab_all = set(freq_pos) | set(freq_neg)
rows = []
for w in vocab_all:
    if w in STOP or len(w) < 3:
        continue
    p = freq_pos[w] / n_pos
    n = freq_neg[w] / n_neg
    total = freq_pos[w] + freq_neg[w]
    if total < 50:  # ignore rare words
        continue
    rows.append((w, p - n, freq_pos[w], freq_neg[w], total))
diff_df = pd.DataFrame(rows, columns=['word', 'pos_minus_neg_rate', 'pos_count', 'neg_count', 'total'])

print('Top 25 words skewed toward POSITIVE class:')
print(diff_df.sort_values('pos_minus_neg_rate', ascending=False).head(25).to_string(index=False))
print()
print('Top 25 words skewed toward NEGATIVE class:')
print(diff_df.sort_values('pos_minus_neg_rate', ascending=True).head(25).to_string(index=False))

## 6. Vocabulary size & coverage

In [ ]:
all_freq = freq_pos + freq_neg
V = len(all_freq)
print(f'Raw vocabulary size (lowercased a-z tokens): {V}')
for cutoff in [1, 2, 5, 10, 50]:
    kept = sum(1 for c in all_freq.values() if c >= cutoff)
    print(f'  >= {cutoff} occurrences: {kept} types ({kept / V:.1%})')

total_tokens = sum(all_freq.values())
print(f'Total tokens in train: {total_tokens:,}')
print(f'Type/token ratio: {V / total_tokens:.4f}')

## 7. Spam tells — quick targeted searches

Without committing to a spam classifier yet, count how often candidate spam markers appear, *broken down by declared label*. If they appear at similar rates in both classes, that's consistent with the brief's claim that spam is mixed evenly.

In [ ]:
MARKERS = {
    'has_subject_header': r'(?im)^\s*subject\s*:',
    'has_url':            r'https?://|www\.',
    'has_email_addr':     r'[\w.\-]+@[\w.\-]+',
    'has_currency':       r'[\$\xa3\u20ac]\s*\d|\busd\b|\bdollars?\b',
    'has_click_here':     r'\bclick\s+here\b',
    'has_unsubscribe':    r'\bunsubscribe\b',
    'has_enron':          r'\benron\b',
    'has_phone_like':     r'\b\d{3}[\s\-.]\d{3}[\s\-.]\d{4}\b',
    'mentions_movie':     r'\b(film|movie|cinema|director|actor|actress|scene|plot|character)\b',
}

marker_df = pd.DataFrame({
    name: train['text'].astype(str).str.contains(pat, regex=True, case=False, na=False)
    for name, pat in MARKERS.items()
})
marker_df['label'] = train['label'].values
rates = marker_df.groupby('label').mean().T
rates.columns = [f'label={c}' for c in rates.columns]
rates['overall'] = marker_df.drop(columns='label').mean()
rates.style.format('{:.1%}')

Read this table carefully:
- The spam markers (`has_subject_header`, `has_url`, `has_enron`, `has_unsubscribe`, …) should appear at **roughly equal rates in both classes** if the brief's "evenly mixed" claim holds.
- The review marker (`mentions_movie`) should appear at **lower** rate than 100% — the gap is roughly the spam contamination fraction.

## 8. Side-by-side: a couple of likely-spam vs likely-review samples

Quick visual confirmation that the markers above identify what we think they identify. *Not* a classifier — just sanity.

In [ ]:
looks_like_spam = train['text'].astype(str).str.contains(
    r'(?im)^\s*subject\s*:|\benron\b|\bunsubscribe\b', regex=True, na=False
)
looks_like_review = train['text'].astype(str).str.contains(
    r'\b(film|movie|director|plot)\b', regex=True, case=False, na=False
) & ~looks_like_spam

print('Looks-like-spam rate by declared label:')
print(looks_like_spam.groupby(train['label']).mean())
print()
print('Looks-like-review rate by declared label:')
print(looks_like_review.groupby(train['label']).mean())

In [ ]:
print('=== Two likely-spam docs sitting under label=1 ===')
show_samples(train[looks_like_spam & (train.label == 1)].assign(label=1), label=1, n=2, seed=1)
print('=== Two likely-spam docs sitting under label=0 ===')
show_samples(train[looks_like_spam & (train.label == 0)].assign(label=0), label=0, n=2, seed=1)
print('=== Two likely-review docs under label=1 ===')
show_samples(train[looks_like_review & (train.label == 1)].assign(label=1), label=1, n=2, seed=1)
print('=== Two likely-review docs under label=0 ===')
show_samples(train[looks_like_review & (train.label == 0)].assign(label=0), label=0, n=2, seed=1)

## 9. Notes-to-self (filled in after running the cells above)

> Edit this cell after reading the outputs. Future-me uses these notes when designing `02_baseline_wordlist`, `03_spam_handling`, and `04_models`.

**Class balance.** _e.g. ~50/50 in train and val? imbalance to flag?_

**What spam looks like in this corpus.** _e.g. Enron-style emails: `Subject:` header, business/finance vocabulary, occasional `<NUM>`-heavy lines, sometimes URLs, names of companies and people. Distinctly different vocabulary from movie reviews._

**Observed rate of spam markers per class.** _Roughly equal across labels 0 and 1, consistent with the brief's claim that spam is mixed evenly._

**Length distribution.** _Bimodal? If yes, length is a useful auxiliary signal but probably not sufficient on its own — some real reviews are long, some spam is short._

**Vocabulary size.** _V ≈ ?, with a long tail. Plan to cap at min_df ≥ 2 or 5 in `02_*` to denoise._

**Stopwords / negation.** _Confirmed `not`, `no`, `never` are in the default NLTK stopword list — DO NOT remove them when building sentiment features (W02_L03 / Tripwire §8)._

**Decisions to make in `02_*` (BoW baseline + word-list classifier):**
- Tokenizer: NLTK regex or simple `[a-z']+`. Lowercase. Map digits to `<NUM>` (W02_L03).
- Reduced stopword list that retains negations.
- BoW frequency vs binary — try both; the lecture covers both.
- Word-list construction by greatest-frequency-difference (W02_L04). Expect leakage of spam-vocabulary into both lists; that motivates `03_*`.

**Decisions to make in `03_*` (spam handling):**
- Approach A: TF-IDF + cosine-to-class-centroid threshold. The threshold sweep curve is part of the report (W03_L05 PR-tradeoff framing).
- Approach B: word-list filter via L_M = L+ ∪ L−.
- Approach C: NB with a max-posterior threshold τ to emit the dummy label.
- Use validation to pick threshold. Validation labels are 0/1 — to evaluate the spam stage we will need either (a) a self-labelled spam set inside val using the same regex markers we used here, or (b) just measure the impact on downstream sentiment accuracy after spam removal.

**Decisions to make in `04_*` (models):**
- Word-list classifier as the simplest baseline (required, W02_L04).
- Multinomial Naive Bayes (W03_L05) — robust to label noise, generative, MLE.
- Logistic Regression (W03_L05) — discriminative, cross-entropy.
- TF-IDF and averaged Word2vec as alternative feature representations (W05_L09) for comparison.

**Tripwires already on the radar:**
- Don't reorder the test set (brief warns twice).
- Don't strip negations.
- Don't include the dummy label in 2-class metrics — switch to 3×3 confusion matrix once spam stage is in (W02_L04 extended).
